# 05 - Staggered DiD: NYC + Florence

With two treated cities at different dates, use Callaway-Sant'Anna to estimate cohort-specific ATTs. Compare to naive TWFE to show the aggregation bias.

In [ ]:
# Data setup
# Set DATA_FILE to 'city_month_panel_real.parquet' after running build_real_panel.py
DATA_FILE = "city_month_panel_real.parquet"   # real Inside Airbnb data
# DATA_FILE = "city_month_panel.parquet"       # synthetic fallback (not committed)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

panel = pd.read_parquet(DATA_DIR / DATA_FILE)
panel["month"] = pd.to_datetime(panel["month"])

regs = pd.read_csv("../data/regulations.csv", parse_dates=["enforcement_date"])

print(f"Panel: {panel.shape}  |  Cities: {sorted(panel['city'].unique())}")
print(f"Date range: {panel['month'].min().date()} to {panel['month'].max().date()}")

In [ ]:
from linearmodels.panel import PanelOLS

## Setup: assign treatment cohorts

In [ ]:
NYC_TREAT      = pd.Timestamp("2023-09-01")
FLORENCE_TREAT = pd.Timestamp("2023-10-01")

panel["treat_cohort"] = None
panel.loc[panel["city"] == "New York City", "treat_cohort"] = str(NYC_TREAT.date())
panel.loc[panel["city"] == "Florence",      "treat_cohort"] = str(FLORENCE_TREAT.date())
panel["never_treated"] = panel["treat_cohort"].isna()

print("Cohort breakdown:")
print(panel.drop_duplicates("city")[["city","treat_cohort","never_treated"]])

## Naive TWFE (biased benchmark)

In [ ]:
panel["post_any"] = (
    ((panel["city"] == "New York City") & (panel["month"] >= NYC_TREAT)) |
    ((panel["city"] == "Florence")      & (panel["month"] >= FLORENCE_TREAT))
).astype(float)
panel["treated_city"] = panel["city"].isin(["New York City","Florence"]).astype(float)
panel["did_any"] = panel["treated_city"] * panel["post_any"]

twfe_idx = panel.set_index(["city","month"])
twfe = PanelOLS(
    twfe_idx["log_listings"],
    twfe_idx[["did_any"]],
    entity_effects=True, time_effects=True,
).fit(cov_type="clustered", cluster_entity=True)

print(f"TWFE pooled ATT = {twfe.params['did_any']:.4f}")
print("Note: TWFE mixes cohort-specific effects and can use already-treated units as controls.")

## Callaway-Sant'Anna: cohort-specific ATTs

In [ ]:
# Manual C-S: for each cohort, compare to never-treated only
# (cleaner than TWFE which uses other treated cohorts as controls)

cohorts = {
    "NYC (Sep 2023)":      ("New York City", NYC_TREAT),
    "Florence (Oct 2023)": ("Florence",      FLORENCE_TREAT),
}

cs_results = {}
for label, (city, treat_date) in cohorts.items():
    # Treated cohort + never-treated controls only
    sub = panel[(panel["city"] == city) | panel["never_treated"]].copy()
    sub["post"]    = (sub["month"] >= treat_date).astype(float)
    sub["treated"] = (sub["city"] == city).astype(float)
    sub["did"]     = sub["treated"] * sub["post"]
    sub_idx = sub.set_index(["city","month"])

    fe = PanelOLS(
        sub_idx["log_listings"],
        sub_idx[["did"]],
        entity_effects=True, time_effects=True,
    ).fit(cov_type="clustered", cluster_entity=True)

    b  = fe.params["did"]
    ci = fe.conf_int().loc["did"]
    cs_results[label] = (b, ci["lower"], ci["upper"])
    pct = (np.exp(b) - 1) * 100
    print(f"{label:<25}  ATT = {b:+.4f}  [{ci['lower']:+.4f}, {ci['upper']:+.4f}]  ({pct:+.1f}% in levels)")

## Comparison: TWFE vs C-S

In [ ]:
print(f"{'Estimator':<30}  {'ATT':>8}")
print("-" * 42)
print(f"{'TWFE (pooled)':<30}  {twfe.params['did_any']:>8.4f}")
for label, (b, lo, hi) in cs_results.items():
    print(f"{'C-S ' + label:<30}  {b:>8.4f}")
print()
print("If TWFE and C-S diverge, it signals heterogeneous treatment effects across cohorts.")
print("C-S is the preferred estimate when effects differ by city or over time.")